<a href="https://colab.research.google.com/github/jasonkwh/mario-snes-cnn-ppo/blob/main/trained_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/jasonkwh/mario-snes-cnn-ppo.git
%cd mario-snes-cnn-ppo
%pip install -U gymnasium stable-retro stable-baselines3 opencv-python

In [ ]:
import gc
import sys
import subprocess
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecTransposeImage
from stable_baselines3.common.callbacks import CheckpointCallback
from cleanup import close_env
from environment import make_env

from config import (
    GAME_NAME, STATE_NAME, MODEL_NAME, RECORD_VIDEO,
    CHECKPOINT_DIR, VIDEO_DIR, MONITOR_FILENAME,
)

if __name__ == "__main__":
    if 'env' in globals():
        close_env(env)
        del env

    gc.collect()

    result = subprocess.run(
        [sys.executable, "-m", "stable_retro.import", "."],
        check=True,
        capture_output=True,
        text=True,
    )

    print(result.stdout)

    # 3. Vectorize using Lambda factory, Stack, and Transpose for PyTorch CNN
    env = DummyVecEnv([lambda: make_env(GAME_NAME, STATE_NAME, RECORD_VIDEO, VIDEO_DIR, MONITOR_FILENAME)])
    env = VecFrameStack(env, n_stack=4, channels_order="last")
    env = VecTransposeImage(env)

    # Automatically check CUDA availability
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 4. Initialize PPO Agent
    model = PPO(
        policy="CnnPolicy",
        env=env,
        device=device,
        verbose=1,
        learning_rate=0.0001,
        n_steps=2048,
        batch_size=64,
        ent_coef=0.01,
    )

    # 5. Train Policy
    print(f"Training {GAME_NAME} Agent on {device.upper()}...")
    model.learn(
        total_timesteps=1000000,
        callback=CheckpointCallback(
            save_freq=50000,
            save_path=CHECKPOINT_DIR,
            name_prefix=MODEL_NAME,
            verbose=2
        )
    )

    # 6. Save Weights
    model.save(MODEL_NAME)
    print(f"Model saved successfully as '{MODEL_NAME}.zip'")